In [1]:
# stt/config
from dataclasses import dataclass

@dataclass
class STTConfig:
    SAMPLE_RATE = 16000
    CHANNELS = 1
    DTYPE = "float32"

    MODEL_NAME = "tiny"

    CHUNK_SECONDS = 1
    LANGUAGE = None       # Auto detect

In [3]:
#stt/recorder.py
import sounddevice as sd
import soundfile as sf
# from stt.config import STTConfig


class AudioRecorder:

    def __init__(self):
        self.sample_rate = STTConfig.SAMPLE_RATE

    def record(self, seconds: int, output_file="audio.wav"):
        print("🎙 Recording...")

        audio = sd.rec(
            int(seconds * self.sample_rate),
            samplerate=self.sample_rate,
            channels=1,
            dtype=STTConfig.DTYPE,
        )

        sd.wait()

        sf.write(output_file, audio, self.sample_rate)

        print("✅ Saved:", output_file)

        return output_file

In [6]:
# stt/whisper_engin.py
import whisper
# from stt.config import STTConfig


class WhisperEngine:

    def __init__(self):
        print("Loading Whisper Tiny...")
        self.model = whisper.load_model(STTConfig.MODEL_NAME)
        print("Model Loaded!")

    def transcribe(self, audio_path: str):

        result = self.model.transcribe(
            audio_path,
            language=STTConfig.LANGUAGE,
        )

        return result

In [1]:
from pathlib import Path

MODEL_DIR = Path("stt/models/indicconformer")

SAMPLE_RATE = 16000

NUM_CHANNELS = 1

CHUNK_SIZE = int(0.1 * SAMPLE_RATE)

In [2]:
import sherpa_onnx

from stt.config import MODEL_DIR


class IndicRecognizer:

    def __init__(self):

        self.recognizer = sherpa_onnx.OnlineRecognizer.from_transducer(
            encoder=str(MODEL_DIR / "encoder.onnx"),
            decoder=str(MODEL_DIR / "decoder.onnx"),
            joiner=str(MODEL_DIR / "joiner.onnx"),
            tokens=str(MODEL_DIR / "tokens.txt"),
            num_threads=2,
            sample_rate=16000,
            feature_dim=80,
        )

    def create_stream(self):
        return self.recognizer.create_stream()

ImportError: libonnxruntime.so: cannot open shared object file: No such file or directory

In [3]:
import sherpa_onnx

print("Sherpa-ONNX version:", sherpa_onnx.version)
print("ONNX Runtime version:", sherpa_onnx.onnxruntime_version)
print("Git SHA:", sherpa_onnx.git_sha1)

Sherpa-ONNX version: 1.13.7
ONNX Runtime version: 1.27.1
Git SHA: 917bed95
